# receipt-generator: Google Spreadsheet から領収書 PDF を作る

スプレッドシートの 1 行を 1 枚の領収書として、PDF を Google Drive に保存します。
指定した Drive の作業フォルダに、コード、`mapping.json`、出力 PDF をまとめて保存します。

上から順に実行してください。
1. 設定を書き換えます。
2. Drive をマウントし、receipt-generator をインストールします。
3. Google アカウントでログインします（スプレッドシートの読み取りに使います）。
4. 自動作成された `mapping.json` とシートの見出しを確認します。
5. 内容をプレビューし、問題がなければ PDF を作ります。

## 1. 設定

In [ ]:
# 読み込むスプレッドシートの URL
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/XXXXXXXX/edit"
# シート（タブ）名。None なら先頭のシート
WORKSHEET = None

# Drive 上の作業フォルダ。自由に変更できます
# code、mapping.json、印影画像、出力 PDF をこの中に置きます
DRIVE_DIR = "/content/drive/MyDrive/receipt-generator"
MAPPING_FILE = "mapping.json"

## 2. Drive のマウントとインストール

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

drive_dir = Path(DRIVE_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)
REPO_DIR = str(drive_dir / "code")
REPO_URL = "https://github.com/ytuesugi/receipt-generator.git"
REPO_BRANCH = "main"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        check=True,
    )

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "libharfbuzz-subset0"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)
sys.path.insert(0, f"{REPO_DIR}/src")
print("receipt-generator:", REPO_DIR)

## 3. Google アカウントでログイン

スプレッドシートを読むために、Colab に Google アカウントへのアクセスを許可します。

In [ ]:
import google.auth
import gspread
from google.colab import auth

auth.authenticate_user()
credentials, _ = google.auth.default()
gc = gspread.authorize(credentials)

## 4. mapping.json の準備

初めて実行したときは、テンプレートから `mapping.json` を作業フォルダに自動作成します。

1. このセルの次にある `mapping.json` の作成セルを実行します。
2. 画面左側のファイルアイコン（フォルダの形）を押します。
3. `drive` → `MyDrive` → `DRIVE_DIR` で指定した作業フォルダを開きます。
4. `mapping.json` をダブルクリックし、開いたエディタで編集します。表示されない場合は、ファイル欄の更新アイコンを押します。
5. 編集内容を保存し、Notebook に戻ります。

下に表示されるシートの見出しに合わせて、`"column"` の値を書き換えてください。
印影を入れる場合は、画像を同じフォルダに置き、`issuer.stamp_file` にファイル名を書きます。

In [ ]:
import json
from pathlib import Path

drive_dir = Path(DRIVE_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)
mapping_path = drive_dir / MAPPING_FILE

if mapping_path.exists():
    print(f"既存の {mapping_path} を使います。")
else:
    template_path = Path(REPO_DIR) / "notebooks" / MAPPING_FILE
    mapping = json.loads(template_path.read_text(encoding="utf-8"))
    mapping["source"] = {"type": "google_sheets", "url": SPREADSHEET_URL, "worksheet": WORKSHEET}
    mapping_path.write_text(json.dumps(mapping, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{mapping_path} に保存しました。シートに合わせて編集してから次へ進んでください。")

spreadsheet = gc.open_by_url(SPREADSHEET_URL)
worksheet = spreadsheet.worksheet(WORKSHEET) if WORKSHEET else spreadsheet.sheet1
print(f"シート「{worksheet.title}」の見出し:")
for header in worksheet.row_values(1):
    print(" -", header)

## 5. プレビュー

PDF はまだ作りません。宛名・明細・合計を確認してください。`mapping.json` を直したら、このセルだけ再実行すれば反映されます。

In [ ]:
import pandas as pd

from receipt_gen import ConfigError
from receipt_gen.sheet import build_config_from_mapping, preview_rows

try:
    result = build_config_from_mapping(
        mapping_path,
        source_override=SPREADSHEET_URL,
        worksheet=WORKSHEET,
        gspread_client=gc,
    )
except ConfigError as error:
    raise SystemExit(f"設定またはシートに問題があります: {error}")

for note in result.skipped:
    print("Skipped", note)
preview = pd.DataFrame(preview_rows(result.config))
print(f"{len(preview)} 件、合計 ¥{preview['total'].sum():,}")
preview

## 6. PDF の生成

`DRIVE_DIR/output/日時/` に保存します。

In [ ]:
from receipt_gen import generate_receipts_from_config

output_files = generate_receipts_from_config(
    result.config,
    base_dir=result.base_dir,
    output_root=drive_dir / "output",
    show_progress=True,
)
output_dir = output_files[0].parent
print(f"{len(output_files)} 件を {output_dir} に保存しました。")

## 7. （任意）ZIP でダウンロード

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(f"/content/{output_dir.name}", "zip", output_dir)
files.download(archive)